## Technical Lesson: LLMs

### Step 0: Install libraries

In [26]:
# ! pip install torch; huggingface_hub[hf_xet]

### Step 1: Import libraries

In [27]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

from transformers import logging
logging.set_verbosity_error()

### Step 2: Define Prediction

In [28]:
def predict_masked_token(masked_text: str) -> str:
    """
    Predict the masked token in a given sentence using the 'bert-base-uncased' model.
    The input sentence must contain exactly one '[MASK]' token.

    Args:
        masked_text (str): A sentence with one '[MASK]' token.

    Returns:
        str: The predicted token that replaces the '[MASK]'.
    """
    # Load the BERT tokenizer and model
    model_name_bert = "bert-base-uncased"
    tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)
    model_bert = AutoModelForMaskedLM.from_pretrained(model_name_bert)

    # Tokenize the input sentence
    inputs = tokenizer_bert(masked_text, return_tensors="pt")

    # Run the model to get logits for the masked token
    with torch.no_grad():
        logits = model_bert(**inputs).logits

    # Find the index of the [MASK] token
    mask_token_index = (inputs["input_ids"] == tokenizer_bert.mask_token_id)[0].nonzero(as_tuple=True)[0]

    # Extract the logits for the masked position and pick the token with the highest score
    mask_logits = logits[0, mask_token_index]
    predicted_token_id = mask_logits.argmax(dim=1)

    # Decode the token ID to a human-readable string
    predicted_token = tokenizer_bert.decode(predicted_token_id)
    return predicted_token.strip()



### Step 3: Define Generate Text

In [29]:
def generate_text(prompt: str) -> str:
    """
    Generate text using the 'distilgpt2' model given an input prompt.

    Args:
        prompt (str): The prompt text to base generation on.

    Returns:
        str: The generated text.
    """
    # Load the GPT tokenizer and model
    model_name_gpt = "distilgpt2"
    tokenizer_gpt = AutoTokenizer.from_pretrained(model_name_gpt)
    model_gpt = AutoModelForCausalLM.from_pretrained(model_name_gpt)

    # Encode the prompt text
    inputs = tokenizer_gpt.encode(prompt, return_tensors="pt")

    # Generate text based on the prompt
    with torch.no_grad():
        generated_ids = model_gpt.generate(inputs, max_length=50, num_return_sequences=1)

    # Decode the generated token IDs into text
    generated_text = tokenizer_gpt.decode(generated_ids[0], skip_special_tokens=True)
    return generated_text.strip()

### Step 4: Get input (two ways)

In [30]:
if __name__ == "__main__":
    # Sample input for the BERT-based masked language model
    sample_masked_sentence = "Artificial intelligence is [MASK] the world by."
    predicted_token = predict_masked_token(sample_masked_sentence)
    print("BERT predicted token:", predicted_token)

    # Sample input for the GPT auto-regressive model
    sample_prompt = "Artificial intelligence is transforming the world by"
    generated_text = generate_text(sample_prompt)
    print("GPT generated text:", generated_text)

BERT predicted token: changing
GPT generated text: Artificial intelligence is transforming the world by creating a new kind of artificial intelligence.
